# 分词 - 练习

关于 [tokenization 视频](https://www.youtube.com/watch?v=zduSFxRajkE)中练习的说明。<br>
改编自 [github.com/karpathy/minbpe/exercise.md](https://github.com/karpathy/minbpe/blob/master/exercise.md)。

1. 观看 YouTube 上的 [tokenization 视频](https://www.youtube.com/watch?v=zduSFxRajkE)
2. 回来解决这些练习，提升自己 :)

**打造你自己的 GPT-4 Tokenizer！**

<style>
/* Keep notebook content printable without horizontal clipping. */
.jp-OutputArea-output img,
.jp-RenderedImage img,
img {
  max-width: 100% !important;
  height: auto !important;
}

.jp-Cell,
.jp-InputArea,
.jp-OutputArea-output,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.jp-InputArea-editor pre,
.jp-RenderedText pre,
.jp-OutputArea-output pre,
.jp-OutputArea-output code,
.highlight pre,
.input_area pre,
.output_area pre,
.output_subarea pre,
.output_text pre,
.output_stream pre,
pre,
code {
  white-space: pre-wrap !important;
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.rendered_html table,
.jp-RenderedHTMLCommon table {
  max-width: 100% !important;
}
</style>


### 第 1 步

编写 `BasicTokenizer` 类，包含以下三个核心函数：

- `def train(self, text, vocab_size, verbose=False)`
- `def encode(self, text)`
- `def decode(self, ids)`

用你喜欢的任意文本训练你的 tokenizer，并可视化合并后的 token。<br>
**它们看起来合理吗？**<br><br>一个你可能想用的默认测试是文本文件 `taylorswift.txt`。

In [2]:
# TODO

### 第 2 步

将你的 `BasicTokenizer` 转换为 `RegexTokenizer`，后者接收一个 regex 模式并完全按照 GPT-4 的方式切分文本。<br>
像之前一样分别处理各部分，然后拼接结果。<br>
重新训练你的 tokenizer，并比较前后的结果。<br><br>
你应该会看到，现在不会再有跨越类别的 token（数字、字母、标点、多于一个空格）。<br><br><br>使用 GPT-4 的模式：

```
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```


In [ ]:
# TODO

### 第 3 步

现在你可以加载 GPT-4 tokenizer 的合并结果，并展示你的 tokenizer 在 `encode` 和 `decode` 上产生完全一致的结果，与 [tiktoken](https://github.com/openai/tiktoken) 匹配。

```
# match this
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids) # get the same text back
```

遗憾的是，你会遇到两个问题：

1. 从 GPT-4 tokenizer 中恢复原始的合并并不简单。你可以很容易地恢复我们这里称之为 `vocab` 的内容，也就是他们称为 `enc._mergeable_ranks` 并存储起来的东西。你可以随意复制粘贴 [`recover_merges`](https://github.com/karpathy/minbpe/blob/master/minbpe/gpt4.py) 中的 `minbpe/gpt4.py` 函数，它接收这些 rank 并返回原始的合并。如果你想了解这个函数的工作原理，请阅读[这个](https://github.com/openai/tiktoken/issues/60)和[这个](https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306)。基本上，在某些条件下，只存储父节点（及其 rank）并丢弃哪些子节点合并到某个父节点的精确细节就足够了。
2. 其次，GPT-4 tokenizer 出于某种原因对其原始字节做了置换。它把这个置换存储在 mergeable ranks 的前 $256$ 个元素中，所以你可以用 `byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}` 相对简单地恢复这个字节重排。在你的 encode 和 decode 中，你都需要相应地对字节进行重排。

In [ ]:
# TODO

### 第 4 步

*（可选、令人头疼、用处不太明显）*<br><br>添加处理特殊 token 的能力。<br>
这样你就能在存在特殊 token 时也匹配 tiktoken 的输出，例如：

```
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("<|endoftext|>hello world", allowed_special="all")
```

如果没有 `allowed_special`，tiktoken 会报错。

In [ ]:
# TODO

### 第 5 步

**如果你走到这一步，你已经是 LLM 分词方面的专家了！**<br><br>
遗憾的是，你还没有*完全*完成，因为 OpenAI 之外的许多 LLM（例如 Llama、Mistral）使用的是 [sentencepiece](https://github.com/google/sentencepiece)。<br>
主要区别在于 sentencepiece 直接在 Unicode 码点上运行 BPE，而不是在 UTF-8 编码的字节上。<br><br>
你可以自行探索 sentencepiece（祝你好运，它不太美观），<br>
如果你真的有时间并愿意承受这份折磨，进阶目标是重写你的 BPE，使其在 Unicode 码点上运行，并匹配 Llama 2 tokenizer。

In [ ]:
# TODO